# Financial Runway Synthetic Data Generator

This notebook builds a local Gradio app that generates fictional financial profiles with multiple Ollama models and prompt strategies. The generated values are validated before Python calculates the expected financial runway.

> **Privacy and safety:** The generated profiles are fictional and should not contain real personal information. Results are educational test data, not financial advice.

## 1. Local setup

Install [Ollama](https://ollama.com/), make sure it is running, and download at least one model. This project works well with the following smaller models:

```text
ollama pull llama3.2
ollama pull phi3
ollama pull deepseek-r1:1.5b
```

Run the next cell to install the Python dependencies into the notebook's active environment.

In [ ]:
%pip install -q ollama pandas gradio

## 2. Imports and configuration

These imports provide the three main parts of the app: Ollama generates profiles locally, pandas organizes them into a table, and Gradio creates the browser interface. The supporting modules parse JSON, validate numbers, and create the downloadable CSV file.

In [ ]:
import json
import math
import tempfile
from pathlib import Path
from typing import Any

import gradio as gr
import ollama
import pandas as pd

PREFERRED_MODELS = ["llama3.2:latest", "phi3:latest", "deepseek-r1:1.5b"]
MAX_ROWS = 30

PROFILE_FIELDS = [
    "household_type",
    "employment_situation",
    "monthly_income",
    "monthly_expenses",
    "cash_savings",
    "investments",
    "debts",
    "financial_goal",
]

## 3. Discover available Ollama models

The `get_installed_models()` function calls `ollama.list()` and returns the names of the models installed locally. It places the preferred models first so they appear first in the Gradio choices.

In [ ]:
def get_installed_models() -> list[str]:
    """Return locally installed Ollama model names, with preferred models first."""
    try:
        response = ollama.list()
        raw_models = response.get("models", []) if isinstance(response, dict) else response.models
        installed = []
        for item in raw_models:
            if isinstance(item, dict):
                name = item.get("model") or item.get("name")
            else:
                name = getattr(item, "model", None) or getattr(item, "name", None)
            if name:
                installed.append(name)

        preferred = [name for name in PREFERRED_MODELS if name in installed]
        return preferred + sorted(name for name in installed if name not in preferred)
    except Exception:
        return []


AVAILABLE_MODELS = get_installed_models()
AVAILABLE_MODELS

## 4. Prompt strategies

The `PROMPT_STRATEGIES` dictionary stores four different generation instructions. Selecting more than one strategy produces a mixture of typical profiles, diverse life situations, financial stress tests, and edge cases.

In [ ]:
PROMPT_STRATEGIES = {
    "Typical households": (
        "Create plausible, everyday household profiles with common income, spending, "
        "savings, debt, and financial goals."
    ),
    "Diverse life situations": (
        "Vary household structure, career stage, employment stability, income level, "
        "expenses, savings habits, and financial goals. Avoid stereotypes."
    ),
    "Financial stress tests": (
        "Create difficult but realistic cases such as irregular income, high expenses, "
        "limited savings, substantial debt, or a recent job change."
    ),
    "Edge cases": (
        "Create unusual but valid profiles that test boundary conditions, including "
        "very short or very long runway and income close to monthly expenses."
    ),
}

DEFAULT_STRATEGIES = ["Typical households", "Diverse life situations"]

## 5. Build a structured generation prompt

The `build_prompt()` function converts the requested row count and strategy into model instructions. It defines the required fields, numeric rules, privacy requirements, and exact JSON response shape.

In [ ]:
def build_prompt(row_count: int, strategy_name: str) -> str:
    """Create instructions for one model-generation batch."""
    strategy = PROMPT_STRATEGIES[strategy_name]
    return f"""
You generate privacy-safe, fictional financial profiles for software testing.
Return exactly {row_count} profiles as valid JSON using this shape:
{{"profiles": [{{
  "household_type": "short fictional household description",
  "employment_situation": "short employment description",
  "monthly_income": 0,
  "monthly_expenses": 0,
  "cash_savings": 0,
  "investments": 0,
  "debts": 0,
  "financial_goal": "short goal"
}}]}}

Rules:
- All money fields must be non-negative JSON numbers in US dollars.
- monthly_expenses must be greater than zero.
- Use plausible values and internally consistent profiles.
- Do not include names, addresses, account numbers, or other identifying information.
- Do not calculate runway and do not add fields outside the schema.
- Return JSON only, with no markdown fences or commentary.

Diversity strategy: {strategy}
""".strip()

## 6. Call Ollama and parse JSON

The `generate_profile_batch()` function sends the constructed prompt to the selected local Ollama model. It requests JSON output, parses the response, and verifies that the response contains a list named `profiles`.

In [ ]:
def _message_content(response: Any) -> str:
    """Read message text from dictionary-style or object-style Ollama responses."""
    if isinstance(response, dict):
        return response["message"]["content"]
    return response.message.content


def generate_profile_batch(model: str, strategy: str, row_count: int) -> list[dict]:
    """Ask one local model to generate one batch of fictional profiles."""
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": build_prompt(row_count, strategy)}],
        format="json",
        options={"temperature": 0.8},
    )
    payload = json.loads(_message_content(response))
    profiles = payload.get("profiles")
    if not isinstance(profiles, list):
        raise ValueError("The model response did not contain a 'profiles' list.")
    return profiles

## 7. Validate profiles and calculate runway

The `validate_and_enrich()` function checks every required field and converts the money values to valid non-negative numbers. Python then calculates cash flow, accessible assets, runway months, and the runway category instead of asking the model to perform the arithmetic.

The same definition used by the earlier Financial Runway Assistant is retained:

```text
accessible assets = cash savings + investments
runway months = accessible assets / monthly expenses
```

In [ ]:
def _valid_money(value: Any) -> float:
    """Convert a finite, non-negative value to a rounded float."""
    if isinstance(value, bool):
        raise ValueError("Boolean values are not valid money amounts.")
    amount = float(value)
    if not math.isfinite(amount) or amount < 0:
        raise ValueError("Money values must be finite and non-negative.")
    return round(amount, 2)


def runway_category(months: float) -> str:
    """Convert runway months into a simple educational category."""
    if months < 3:
        return "Very short"
    if months < 6:
        return "Short"
    if months < 12:
        return "Moderate"
    return "Extended"


def validate_and_enrich(profile: dict, model: str, strategy: str) -> dict:
    """Validate a model-created profile and add deterministic expected results."""
    missing = [field for field in PROFILE_FIELDS if field not in profile]
    if missing:
        raise ValueError(f"Missing fields: {', '.join(missing)}")

    record = {
        "household_type": str(profile["household_type"]).strip(),
        "employment_situation": str(profile["employment_situation"]).strip(),
        "monthly_income": _valid_money(profile["monthly_income"]),
        "monthly_expenses": _valid_money(profile["monthly_expenses"]),
        "cash_savings": _valid_money(profile["cash_savings"]),
        "investments": _valid_money(profile["investments"]),
        "debts": _valid_money(profile["debts"]),
        "financial_goal": str(profile["financial_goal"]).strip(),
    }
    if record["monthly_expenses"] <= 0:
        raise ValueError("Monthly expenses must be greater than zero.")
    if not all(record[field] for field in ("household_type", "employment_situation", "financial_goal")):
        raise ValueError("Text fields cannot be empty.")

    accessible_assets = record["cash_savings"] + record["investments"]
    runway_months = accessible_assets / record["monthly_expenses"]
    record.update({
        "monthly_cash_flow": round(record["monthly_income"] - record["monthly_expenses"], 2),
        "accessible_assets": round(accessible_assets, 2),
        "runway_months": round(runway_months, 1),
        "runway_category": runway_category(runway_months),
        "generated_by": model,
        "prompt_strategy": strategy,
    })
    return record

## 8. Coordinate models, strategies, validation, and export

The `generate_dataset()` function divides the requested profiles among the selected models and strategies. It retries short batches, rejects invalid records, removes duplicates, creates profile IDs, and exports the numeric dataset as a CSV file.

In [ ]:
def _allocate_rows(total: int, task_count: int) -> list[int]:
    """Distribute a total as evenly as possible across generation tasks."""
    base, remainder = divmod(total, task_count)
    return [base + (index < remainder) for index in range(task_count)]


MONEY_COLUMNS = [
    "monthly_income", "monthly_expenses", "cash_savings",
    "investments", "debts", "monthly_cash_flow", "accessible_assets",
]

DISPLAY_NAMES = {
    "profile_id": "Profile ID",
    "household_type": "Household Type",
    "employment_situation": "Employment Situation",
    "monthly_income": "Monthly Income",
    "monthly_expenses": "Monthly Expenses",
    "cash_savings": "Cash Savings",
    "investments": "Investments",
    "debts": "Debts",
    "financial_goal": "Financial Goal",
    "monthly_cash_flow": "Monthly Cash Flow",
    "accessible_assets": "Accessible Assets",
    "runway_months": "Runway (Months)",
    "runway_category": "Runway Category",
    "generated_by": "Generated By",
    "prompt_strategy": "Prompt Strategy",
}


def format_for_display(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Create a readable UI copy without changing the numeric CSV data."""
    display = dataframe.copy()
    for column in MONEY_COLUMNS:
        if column in display.columns:
            display[column] = display[column].map(lambda value: f"${value:,.0f}")
    return display.rename(columns=DISPLAY_NAMES)


def generate_dataset(
    row_count: int, selected_models: list[str], selected_strategies: list[str]
) -> tuple[pd.DataFrame, str | None, str]:
    """Generate, validate, combine, and export fictional financial profiles."""
    row_count = int(row_count)
    if not 1 <= row_count <= MAX_ROWS:
        return pd.DataFrame(), None, f"Choose between 1 and {MAX_ROWS} rows."
    if not selected_models:
        return pd.DataFrame(), None, "Select at least one installed Ollama model."
    if not selected_strategies:
        return pd.DataFrame(), None, "Select at least one prompt strategy."

    tasks = [(model, strategy) for model in selected_models for strategy in selected_strategies]
    tasks = tasks[:row_count]  # Every active task receives at least one requested row.
    allocations = _allocate_rows(row_count, len(tasks))
    valid_records, errors = [], []

    for (model, strategy), batch_size in zip(tasks, allocations):
        accepted_for_task = 0
        attempts = 0
        while accepted_for_task < batch_size and attempts < 3:
            attempts += 1
            remaining = batch_size - accepted_for_task
            try:
                profiles = generate_profile_batch(model, strategy, remaining)
                for profile in profiles[:remaining]:
                    try:
                        valid_records.append(validate_and_enrich(profile, model, strategy))
                        accepted_for_task += 1
                    except (TypeError, ValueError) as exc:
                        errors.append(f"{model}: rejected one record ({exc})")
            except Exception as exc:
                errors.append(f"{model} with {strategy}: {exc}")
                break

    if not valid_records:
        detail = errors[0] if errors else "No profiles were returned."
        return pd.DataFrame(), None, f"Generation failed. {detail}"

    dataframe = pd.DataFrame(valid_records)
    dedupe_columns = ["household_type", "employment_situation", "monthly_income", "monthly_expenses", "financial_goal"]
    dataframe = dataframe.drop_duplicates(subset=dedupe_columns).head(row_count).reset_index(drop=True)
    dataframe.insert(0, "profile_id", [f"FR-{number:03d}" for number in range(1, len(dataframe) + 1)])

    with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix="financial_runway_", delete=False, newline="", encoding="utf-8") as output_file:
        dataframe.to_csv(output_file.name, index=False)
        csv_path = output_file.name

    status = f"Generated {len(dataframe)} validated fictional profiles using {len(selected_models)} model(s)."
    if len(dataframe) < row_count:
        status += f" Requested {row_count}; some outputs were invalid or duplicated."
    if errors:
        status += f" Skipped {len(errors)} invalid batch or record(s)."
    return format_for_display(dataframe), csv_path, status

## 9. Gradio interface

The Gradio `Blocks` layout connects the row-count, model, and strategy controls to `generate_dataset()`. The preview uses readable headings and formatted dollar values, while the download keeps the original numeric columns for later analysis.

In [ ]:
default_models = AVAILABLE_MODELS[:2]

with gr.Blocks(title="Financial Runway Synthetic Data Generator") as app:
    gr.Markdown("# Financial Runway Synthetic Data Generator")
    gr.Markdown(
        "Generate privacy-safe fictional financial profiles with local Ollama models. "
        "Python validates each record and calculates its expected financial runway. "
        "This tool creates educational test data, not financial advice."
    )

    if not AVAILABLE_MODELS:
        gr.Markdown("⚠️ No Ollama models were detected. Start Ollama, pull a model, and rerun the notebook.")

    with gr.Row():
        row_count_input = gr.Slider(
            minimum=1, maximum=MAX_ROWS, step=1, value=12, label="Number of profiles"
        )
        model_input = gr.CheckboxGroup(
            choices=AVAILABLE_MODELS, value=default_models, label="Local Ollama models"
        )

    strategy_input = gr.CheckboxGroup(
        choices=list(PROMPT_STRATEGIES),
        value=DEFAULT_STRATEGIES,
        label="Prompt strategies",
    )
    generate_button = gr.Button("Generate dataset", variant="primary")
    status_output = gr.Markdown()
    table_output = gr.Dataframe(label="Validated synthetic profiles", interactive=False)
    download_output = gr.File(label="Download CSV")

    generate_button.click(
        fn=generate_dataset,
        inputs=[row_count_input, model_input, strategy_input],
        outputs=[table_output, download_output, status_output],
    )

## 10. Launch the application

The final cell calls `app.launch()` to start the interface. After Ollama is running, Gradio prints a local browser address, usually `http://127.0.0.1:7860`.

In [ ]:
app.launch()